# Hands-on Exercise 5 — Serve Locally & Query via curl
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 5 (Model Serving)

**Objective:** deploy the `Production` version of your registered model as a local REST API with a
single command, then query it — first successfully, then deliberately with malformed input to observe
MLflow's signature validation.

**Steps (from the slide deck):**
1. Start a local server: `mlflow models serve -m "models:/my-classifier/Production" -p 5001 --env-manager=local`.
2. In a second terminal, run the `curl /invocations` command against it with a real sample row.
3. Confirm the JSON response contains a sensible prediction.
4. Deliberately send a malformed request (wrong column name or missing field) and observe MLflow's validation error.
5. Bonus: write the equivalent Python `requests` call and run it from a script instead of curl.

**Deliverable:** a terminal transcript (or screenshot) showing one successful prediction and one
rejected/invalid request, demonstrating signature validation in action.

> **Prerequisites:**
> - The MLflow Tracking Server must still be running at `http://localhost:5000`.
> - You must have completed Exercise 4, so `models:/my-classifier/Production` exists.
> - `mlflow models serve` needs its own environment build the first time it runs — this can take a
>   minute or two.

> **Recommended approach:** the cleanest way to run this exercise is with **two terminals**:
> Terminal A runs the server (Step 1), Terminal B runs curl (Steps 2–4). The cells below show you
> exactly what to type in each terminal, and also include a notebook-native fallback using
> `subprocess` if you'd rather not leave the notebook.

## Step 1 — Start the server (run this in a TERMINAL, not in this notebook)

```bash
mlflow models serve \
    -m "models:/my-classifier/Production" \
    -p 5001 --env-manager=local
```

Leave that terminal running. Wait until you see a line like `Listening at: http://127.0.0.1:5001`
before moving on to Step 2.

## Step 2 — Query it with curl (run this in a SECOND terminal)

In [ ]:
# Copy this block into your second terminal:

curl_example = '''
curl -X POST http://localhost:5001/invocations \\
    -H "Content-Type: application/json" \\
    -d '{
          "dataframe_split": {
            "columns": ["sepal length (cm)", "sepal width (cm)",
                        "petal length (cm)", "petal width (cm)"],
            "data": [[5.1, 3.5, 1.4, 0.2]]
          }
        }'
'''
print(curl_example)

## Step 3 — Confirm the response
Expect a JSON body like `{"predictions": [0]}`. Paste your terminal's actual response below as a comment for your records.

In [ ]:
# Paste the JSON response you received here, e.g.:
# {"predictions": [0]}


## Step 4 — Send a malformed request and observe the validation error
Try a wrong column name (`sepal_length` instead of `sepal length (cm)`) or a missing field.

In [ ]:
curl_bad_example = '''
curl -X POST http://localhost:5001/invocations \\
    -H "Content-Type: application/json" \\
    -d '{
          "dataframe_split": {
            "columns": ["sepal_length", "sepal_width", "petal_length", "petal_width"],
            "data": [[5.1, 3.5, 1.4, 0.2]]
          }
        }'
'''
print(curl_bad_example)
# Expect an HTTP 400 response with a schema/validation error message,
# because the column names don't match the model's logged signature (Exercise 3).

## Step 5 (Bonus) — Equivalent Python client
Run this cell directly in the notebook once the server from Step 1 is up.

In [ ]:
import requests

url = "http://localhost:5001/invocations"

good_payload = {
    "dataframe_records": [
        {"sepal length (cm)": 5.1, "sepal width (cm)": 3.5,
         "petal length (cm)": 1.4, "petal width (cm)": 0.2}
    ]
}

try:
    resp = requests.post(url, json=good_payload, timeout=10)
    print("Status:", resp.status_code)
    print("Response:", resp.json())
except requests.exceptions.ConnectionError:
    print("Could not reach the server — make sure Step 1's `mlflow models serve` command is running.")

In [ ]:
bad_payload = {
    "dataframe_records": [
        {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
    ]
}

try:
    resp = requests.post(url, json=bad_payload, timeout=10)
    print("Status:", resp.status_code)
    print("Response:", resp.text[:500])
except requests.exceptions.ConnectionError:
    print("Could not reach the server — make sure Step 1's `mlflow models serve` command is running.")

## Optional — Fully notebook-native version (background subprocess)
If you'd rather not use a second terminal at all, this cell launches the server as a background
process from within the notebook, waits for it to become healthy, and then you can run the
`requests` cells above normally. **Remember to shut it down** with the cell at the very end.

In [ ]:
import subprocess, time, requests as _requests

server_proc = subprocess.Popen(
    ["mlflow", "models", "serve", "-m", "models:/my-classifier/Production",
     "-p", "5001", "--env-manager", "local"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

print("Starting server (this can take a minute the first time)...")
for _ in range(10):
    try:
        r = _requests.get("http://localhost:5001/ping", timeout=2)
        if r.status_code == 200:
            print("Server is up!")
            break
    except _requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    print("Server did not become healthy in time — check server_proc.stdout for errors.")

Now re-run the **Step 5 (Bonus)** `requests` cells above — they should succeed.

### Shutdown
Run this when you're done, to stop the background server started in the optional cell above.

In [ ]:
if 'server_proc' in dir():
    server_proc.terminate()
    print("Server stopped.")

---
### ✅ Deliverable checklist
- [ ] Terminal transcript / screenshot of a successful `/invocations` call with a real prediction
- [ ] Terminal transcript / screenshot of a rejected/malformed request showing MLflow's schema
      validation error
- [ ] (Bonus) The equivalent Python `requests` script, executed successfully in this notebook